# Tiny Dreamer Highway — Colab Setup and Smoke Tests

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

This notebook follows the project rule: GitHub is the source of truth for code, and Google Drive stores artifacts such as checkpoints, plots, and videos.

Its role is to combine Colab environment setup with lightweight smoke tests for the package, training helpers, evaluation metrics, and artifact exports.

## Runtime flow

1. Mount Google Drive.
2. Clone or pull the latest repository snapshot from GitHub.
3. Install the package and dependencies.
4. Run a small sanity check before training.
5. Save outputs into the Drive-backed `artifacts/` folder.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO_URL = 'https://github.com/estmon8u/CSC_580_Final_Project.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
WORKTREE = Path('/content/CSC_580_Final_Project')

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'checkpoints', ARTIFACT_ROOT / 'media', ARTIFACT_ROOT / 'logs']:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Artifact root:', ARTIFACT_ROOT)
print('Repository URL:', REPO_URL)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root: /content/drive/MyDrive/CSC_580_Final_Project
Artifact root: /content/drive/MyDrive/CSC_580_Final_Project/artifacts
Repository URL: https://github.com/estmon8u/CSC_580_Final_Project.git


In [2]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi

Updating adfb9de..2119cfe
Fast-forward
 .gitignore                                         |   2 +-
 README.md                                          |   3 +-
 examples/base_experiment.yaml                      |  19 +-
 examples/h100_amp_experiment.yaml                  |  19 +-
 examples/h100_experiment.yaml                      |  19 +-
 examples/h100_screening_experiment.yaml            |  19 +-
 examples/optimized_experiment.yaml                 |  19 +-
 examples/training_run.yaml                         |  19 +-
 notebooks/01_colab_setup_and_smoke_tests.ipynb     | 290 ++++++++++++++++-----
 notebooks/03_colab_baseline_run.ipynb              |  19 +-
 notebooks/05_colab_optimized_run.ipynb             |  19 +-
 notebooks/07_colab_screening_run.ipynb             |  30 ++-
 notebooks/README.md                                |   2 +
 src/tiny_dreamer_highway/__init__.py               |   2 +-
 src/tiny_dreamer_highway/cli.py                    |   5 +-
 src/tiny_dreamer_highway/c

From https://github.com/estmon8u/CSC_580_Final_Project
 * branch            main       -> FETCH_HEAD
   adfb9de..2119cfe  main       -> origin/main


In [3]:
%%bash
set -e
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip --quiet

MARKER=/content/CSC_580_Final_Project/.runtime_restart_required

# Check the numpy version currently active in this runtime's Python
CURRENT_NP=$(python -c "import numpy; print(numpy.__version__)" 2>/dev/null || echo "none")
echo "Runtime numpy: ${CURRENT_NP}"

if [ "${CURRENT_NP}" != "2.0.2" ]; then
  echo "NumPy is ${CURRENT_NP}, not 2.0.2 — force-reinstalling scientific stack from PyPI..."
  # Force download fresh wheels compiled for numpy 2.0 (replaces Colab's pre-built binaries)
  python -m pip install --force-reinstall --quiet "numpy==2.0.2" "scipy==1.14.1"
  echo "restart-required" > "${MARKER}"
  echo ""
  echo ">>> STOP: Restart the Colab runtime now, then rerun from the top. <<<"
else
  echo "NumPy 2.0.2 already active — installing remaining dependencies normally..."
  python -m pip install -e . --quiet
  rm -f "${MARKER}"
  echo "Install complete — no restart required."
fi


Runtime numpy: 2.0.2
NumPy 2.0.2 already active — installing remaining dependencies normally...
Install complete — no restart required.


## Dependency source of truth

This notebook installs the project directly from the repository with `pip install -e .`. All dependency versions come from `pyproject.toml`.

### Why the install cell checks the runtime numpy version

Colab ships its own pre-built `scipy` binary compiled against numpy 1.x. If we simply run `pip install scipy==1.14.1`, pip sees the version already matches and skips the download — leaving the old numpy-1.x-compiled `.so` files in place. Mixing those with numpy 2.0 at import time causes `AttributeError: module 'numpy' has no attribute '_no_nep50_warning'`.

**The fix:** the install cell checks `numpy.__version__` in the *running* Python interpreter (not pip metadata):
- If it is **not** `2.0.2`, it runs `pip install --force-reinstall numpy==2.0.2 scipy==1.14.1` to download fresh PyPI wheels compiled for numpy 2.0, writes a restart marker, and stops.
- If it **is already** `2.0.2`, it runs `pip install -e .` normally (numpy/scipy are already correct), removes the marker, and continues.

**Expected flow on a fresh Colab runtime:**
1. Run install cell → detects numpy 1.x → force-reinstalls → prints STOP message.
2. Restart runtime (*Runtime → Restart runtime*).
3. Re-run all cells from the top → install cell detects numpy 2.0.2 → normal install → no restart needed.
4. Version-check cell passes → continue.


In [4]:
from importlib.metadata import version
from pathlib import Path

restart_marker = Path('/content/CSC_580_Final_Project/.runtime_restart_required')

print('numpy version:', version('numpy'))
print('scipy version:', version('scipy'))
print('gymnasium version:', version('gymnasium'))
print('highway-env version:', version('highway-env'))

if restart_marker.exists():
    raise RuntimeError(
        'The install step changed NumPy or SciPy in this active runtime. '
        'Restart the Colab runtime now, then rerun the notebook from the top. '
        'After restart, the install cell should leave the binary packages unchanged and this check will pass.'
    )

print('\nBinary dependency state looks stable. Safe to continue.')


numpy version: 2.0.2
scipy version: 1.14.1
gymnasium version: 1.2.3
highway-env version: 1.10.2

Binary dependency state looks stable. Safe to continue.


In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.cli import summarize_config

config_path = PROJECT_ROOT / 'examples' / 'base_experiment.yaml'
config = load_experiment_config(config_path)
print(summarize_config(config))

Loaded config for highway-v0 | obs=64x64 | replay_capacity=10000 | sequence_length=8 | batch_size=4


## Reproducibility seed

This cell applies the experiment seed from the config to Python, NumPy, and PyTorch so the Colab smoke tests are more repeatable.

In [6]:
from tiny_dreamer_highway.utils import set_global_seeds

set_global_seeds(config.seed, deterministic_torch=False)
print('Using experiment seed:', config.seed)

Using experiment seed: 7


In [7]:
from tiny_dreamer_highway.envs.highway_factory import make_highway_env

env = make_highway_env(config.env)
if hasattr(env.action_space, 'seed'):
    env.action_space.seed(config.seed)
observation, info = env.reset(seed=config.seed)
action = env.action_space.sample()
next_observation, reward, terminated, truncated, info = env.step(action)
env.close()

print('Observation shape:', getattr(observation, 'shape', None))
print('Action shape:', getattr(action, 'shape', None))
print('Reward:', reward)
print('Episode ended:', bool(terminated or truncated))

Observation shape: (1, 64, 64)
Action shape: (2,)
Reward: 0.635364174346129
Episode ended: False


## Replay warm-start smoke test

This cell exercises milestone M2: collect a small batch of random transitions, then verify that replay batch sampling and sequence sampling both work.

In [8]:
from tiny_dreamer_highway.data.collect_random_rollouts import collect_random_transitions
from tiny_dreamer_highway.data.replay_buffer import ReplayBuffer

replay_buffer = ReplayBuffer(capacity=config.replay.capacity)
added = collect_random_transitions(config.env, replay_buffer, steps=32, seed=config.seed)

batch = replay_buffer.sample_batch(batch_size=config.replay.batch_size)
sequences = replay_buffer.sample_sequences(
    batch_size=config.replay.batch_size,
    sequence_length=config.replay.sequence_length,
)

print('Collected transitions:', added)
print('Replay size:', len(replay_buffer))
print('Batch observation shape:', batch.observations.shape)
print('Batch action shape:', batch.actions.shape)
print('Sequence batch:', len(sequences), 'x', len(sequences[0]))

Collected transitions: 32
Replay size: 32
Batch observation shape: (4, 1, 64, 64)
Batch action shape: (4, 2)
Sequence batch: 4 x 8


## Encoder shape smoke test

This cell starts milestone M3 by pushing a sampled replay batch through the new observation encoder and verifying the latent feature shape.

In [9]:
import torch
from tiny_dreamer_highway.models import ObservationEncoder

observation_tensor = torch.as_tensor(batch.observations)
if observation_tensor.ndim == 3:
    observation_tensor = observation_tensor.unsqueeze(1)

encoder = ObservationEncoder(
    in_channels=observation_tensor.shape[1],
    observation_shape=tuple(observation_tensor.shape[-2:]),
    embedding_dim=config.model.embedding_dim,
)
latent = encoder(observation_tensor)

print('Encoder input shape:', tuple(observation_tensor.shape))
print('Latent embedding shape:', tuple(latent.features.shape))
print('Conv output shape:', encoder.conv_output_shape)

Encoder input shape: (4, 1, 64, 64)
Latent embedding shape: (4, 256)
Conv output shape: (256, 4, 4)


## RSSM shape smoke test

This cell advances milestone M3 by feeding encoder embeddings and replay actions through the RSSM prior and posterior updates and verifying the latent state shapes.

In [10]:
try:
    from tiny_dreamer_highway.models import RecurrentStateSpaceModel
except ImportError:
    from tiny_dreamer_highway.models.rssm import RecurrentStateSpaceModel

action_tensor = torch.as_tensor(batch.actions, dtype=torch.float32)
embedding_tensor = latent.embedding

rssm = RecurrentStateSpaceModel(
    action_dim=action_tensor.shape[-1],
    embedding_dim=config.model.embedding_dim,
    deterministic_dim=config.model.deterministic_dim,
    stochastic_dim=config.model.stochastic_dim,
    hidden_dim=config.model.hidden_dim,
    num_layers=config.model.rssm_num_layers,
)
initial_state = rssm.initial_state(batch_size=action_tensor.shape[0], device=action_tensor.device)
prior_state = rssm.imagine_step(initial_state, action_tensor)
posterior_state = rssm.observe_step(initial_state, action_tensor, embedding_tensor)

print('Action shape:', tuple(action_tensor.shape))
print('Embedding shape:', tuple(embedding_tensor.shape))
print('Prior deterministic shape:', tuple(prior_state.deterministic.shape))
print('Prior stochastic shape:', tuple(prior_state.stochastic.shape))
print('Posterior deterministic shape:', tuple(posterior_state.deterministic.shape))
print('Posterior stochastic shape:', tuple(posterior_state.stochastic.shape))

Action shape: (4, 2)
Embedding shape: (4, 256)
Prior deterministic shape: (4, 128)
Prior stochastic shape: (4, 32)
Posterior deterministic shape: (4, 128)
Posterior stochastic shape: (4, 32)


## Decoder and reward head smoke test

This cell completes the core M3 forward-pass smoke test by decoding the posterior latent state back to an image-shaped tensor and predicting rewards from the same latent features.

In [11]:
try:
    from tiny_dreamer_highway.models import ObservationDecoder, RewardPredictor
except ImportError:
    from tiny_dreamer_highway.models.decoder import ObservationDecoder, RewardPredictor

latent_features = posterior_state.features
latent_dim = config.model.deterministic_dim + config.model.stochastic_dim

decoder = ObservationDecoder(
    latent_dim=latent_dim,
    output_shape=(observation_tensor.shape[1], observation_tensor.shape[2], observation_tensor.shape[3]),
)
reward_predictor = RewardPredictor(
    latent_dim=latent_dim,
    hidden_dim=config.model.reward_hidden_dim,
    num_layers=config.model.reward_num_layers,
)

reconstruction = decoder(latent_features)
predicted_reward = reward_predictor(latent_features)

print('Posterior latent feature shape:', tuple(latent_features.shape))
print('Reconstruction shape:', tuple(reconstruction.shape))
print('Target observation shape:', tuple(observation_tensor.shape))
print('Predicted reward shape:', tuple(predicted_reward.shape))

Posterior latent feature shape: (4, 160)
Reconstruction shape: (4, 1, 64, 64)
Target observation shape: (4, 1, 64, 64)
Predicted reward shape: (4, 1)


## Tiny world-model optimization smoke test

This cell wires the encoder, RSSM, decoder, and reward head into a single forward pass and runs one tiny optimization step on the sampled replay batch.

In [12]:
try:
    from tiny_dreamer_highway.models import TinyWorldModel
except ImportError:
    from tiny_dreamer_highway.models.world_model import TinyWorldModel

try:
    from tiny_dreamer_highway.training import compute_world_model_losses, train_world_model_step
except ImportError:
    from tiny_dreamer_highway.training.world_model_step import (
        compute_world_model_losses,
        train_world_model_step,
    )

set_global_seeds(config.seed, deterministic_torch=False)

world_model = TinyWorldModel(
    observation_shape=(observation_tensor.shape[1], observation_tensor.shape[2], observation_tensor.shape[3]),
    action_dim=action_tensor.shape[-1],
    embedding_dim=config.model.embedding_dim,
    deterministic_dim=config.model.deterministic_dim,
    stochastic_dim=config.model.stochastic_dim,
    hidden_dim=config.model.hidden_dim,
    rssm_num_layers=config.model.rssm_num_layers,
    reward_hidden_dim=config.model.reward_hidden_dim,
    reward_num_layers=config.model.reward_num_layers,
)
optimizer = torch.optim.Adam(world_model.parameters(), lr=1e-3)
reward_targets = torch.as_tensor(batch.rewards, dtype=torch.float32)

with torch.no_grad():
    before_output = world_model(observation_tensor, action_tensor)
    before_losses = compute_world_model_losses(before_output, observation_tensor, reward_targets)

parameter_before = next(world_model.parameters()).detach().clone()
train_output, metrics = train_world_model_step(
    world_model,
    optimizer,
    observation_tensor,
    action_tensor,
    reward_targets,
)
parameter_after = next(world_model.parameters()).detach().clone()

with torch.no_grad():
    after_output = world_model(observation_tensor, action_tensor)
    after_losses = compute_world_model_losses(after_output, observation_tensor, reward_targets)

parameter_delta = (parameter_after - parameter_before).abs().mean().item()
loss_delta = after_losses['total_loss'].item() - before_losses['total_loss'].item()
reconstruction_detached = train_output.reconstruction.detach()
predicted_reward_detached = train_output.predicted_reward.detach()

print('Seed used:', config.seed)
print('Pre-step total loss:', before_losses['total_loss'].item())
print('Post-step total loss:', after_losses['total_loss'].item())
print('Loss delta (post - pre):', loss_delta)
print('Parameter mean abs update:', parameter_delta)
print('Train-step reported total loss:', metrics['total_loss'])
print('Reconstruction shape:', tuple(train_output.reconstruction.shape))
print('Predicted reward shape:', tuple(train_output.predicted_reward.shape))
print('Reconstruction min/max:', float(reconstruction_detached.min()), float(reconstruction_detached.max()))
print('Predicted reward mean/std:', float(predicted_reward_detached.mean()), float(predicted_reward_detached.std()))
print('All losses finite:', bool(torch.isfinite(before_losses['total_loss']) and torch.isfinite(after_losses['total_loss'])))
print('Parameters changed:', parameter_delta > 0.0)

Seed used: 7
Pre-step total loss: 5169.9921875
Post-step total loss: 5994.1279296875
Loss delta (post - pre): 824.1357421875
Parameter mean abs update: 0.0008847593562677503
Train-step reported total loss: 5004.3154296875
Reconstruction shape: (4, 1, 64, 64)
Predicted reward shape: (4, 1)
Reconstruction min/max: -3.4706530570983887 2.807743549346924
Predicted reward mean/std: -0.5385167598724365 0.550163984298706
All losses finite: True
Parameters changed: True


## Short sequence world-model training smoke test

This cell moves beyond single-step training by sampling replay sequences, stacking them into tensors, and running one sequence-level world-model optimization step.

In [13]:
try:
    from tiny_dreamer_highway.training import stack_sequence_batch, train_sequence_world_model_step
except ImportError:
    from tiny_dreamer_highway.training.sequence_world_model_step import (
        stack_sequence_batch,
        train_sequence_world_model_step,
    )

set_global_seeds(config.seed, deterministic_torch=False)

sequence_samples = replay_buffer.sample_sequences(
    batch_size=config.replay.batch_size,
    sequence_length=config.replay.sequence_length,
)
sequence_batch = stack_sequence_batch(sequence_samples)
sequence_observations = torch.as_tensor(sequence_batch.observations)
sequence_actions = torch.as_tensor(sequence_batch.actions, dtype=torch.float32)
sequence_rewards = torch.as_tensor(sequence_batch.rewards, dtype=torch.float32)

sequence_world_model = TinyWorldModel(
    observation_shape=(sequence_observations.shape[2], sequence_observations.shape[3], sequence_observations.shape[4]),
    action_dim=sequence_actions.shape[-1],
    embedding_dim=config.model.embedding_dim,
    deterministic_dim=config.model.deterministic_dim,
    stochastic_dim=config.model.stochastic_dim,
    hidden_dim=config.model.hidden_dim,
    rssm_num_layers=config.model.rssm_num_layers,
    reward_hidden_dim=config.model.reward_hidden_dim,
    reward_num_layers=config.model.reward_num_layers,
)
sequence_optimizer = torch.optim.Adam(sequence_world_model.parameters(), lr=1e-3)
sequence_outputs, sequence_metrics = train_sequence_world_model_step(
    sequence_world_model,
    sequence_optimizer,
    sequence_observations,
    sequence_actions,
    sequence_rewards,
)

print('Seed used:', config.seed)
print('Sequence batch observation shape:', tuple(sequence_observations.shape))
print('Sequence batch action shape:', tuple(sequence_actions.shape))
print('Sequence total loss:', sequence_metrics['total_loss'])
print('Sequence reconstruction loss:', sequence_metrics['reconstruction_loss'])
print('Sequence reward loss:', sequence_metrics['reward_loss'])
print('Sequence steps processed:', len(sequence_outputs))
print('Last-step reconstruction shape:', tuple(sequence_outputs[-1].reconstruction.shape))
print('Last-step reward shape:', tuple(sequence_outputs[-1].predicted_reward.shape))

Seed used: 7
Sequence batch observation shape: (4, 8, 1, 64, 64)
Sequence batch action shape: (4, 8, 2)
Sequence total loss: 6214.4599609375
Sequence reconstruction loss: 6153.056640625
Sequence reward loss: 5.4556193351745605
Sequence steps processed: 8
Last-step reconstruction shape: (4, 1, 64, 64)
Last-step reward shape: (4, 1)


## Imagination, actor, and critic smoke test

This cell seeds imagination from a grounded posterior state, runs a short latent rollout, computes TD-lambda returns, and performs one actor-critic update while checking that world-model parameters stay unchanged.

In [14]:
try:
    from tiny_dreamer_highway.models import Actor, Critic
    from tiny_dreamer_highway.training import imagine_trajectory, td_lambda_returns, train_behavior_step
except ImportError:
    from tiny_dreamer_highway.models.actor import Actor
    from tiny_dreamer_highway.models.critic import Critic
    from tiny_dreamer_highway.training.behavior_learning import (
        imagine_trajectory,
        td_lambda_returns,
        train_behavior_step,
    )

set_global_seeds(config.seed, deterministic_torch=False)

with torch.no_grad():
    grounded_output = world_model(observation_tensor, action_tensor)
    start_state = grounded_output.posterior_state

latent_dim = config.model.deterministic_dim + config.model.stochastic_dim
actor = Actor(
    latent_dim=latent_dim,
    action_dim=action_tensor.shape[-1],
    hidden_dim=config.model.actor_hidden_dim,
    num_layers=config.model.actor_num_layers,
)
critic = Critic(
    latent_dim=latent_dim,
    hidden_dim=config.model.critic_hidden_dim,
    num_layers=config.model.critic_num_layers,
)
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=config.training.actor_lr)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=config.training.critic_lr)

actor_before = next(actor.parameters()).detach().clone()
critic_before = next(critic.parameters()).detach().clone()
world_before = next(world_model.parameters()).detach().clone()

trajectory = imagine_trajectory(
    world_model,
    actor,
    critic,
    start_state,
    horizon=config.training.imagination_horizon,
    )
returns = td_lambda_returns(
    trajectory.rewards,
    trajectory.values,
    bootstrap=trajectory.bootstrap,
    discount=0.99,
    lambda_=0.95,
    )
behavior_metrics = train_behavior_step(
    world_model,
    actor,
    critic,
    actor_optimizer,
    critic_optimizer,
    start_state,
    horizon=config.training.imagination_horizon,
    discount=0.99,
    lambda_=0.95,
    )

actor_after = next(actor.parameters()).detach().clone()
critic_after = next(critic.parameters()).detach().clone()
world_after = next(world_model.parameters()).detach().clone()

print('Seed used:', config.seed)
print('Imagination horizon:', config.training.imagination_horizon)
print('Imagined feature shape:', tuple(trajectory.features.shape))
print('Imagined action shape:', tuple(trajectory.actions.shape))
print('Imagined reward shape:', tuple(trajectory.rewards.shape))
print('Imagined value shape:', tuple(trajectory.values.shape))
print('TD-lambda return shape:', tuple(returns.shape))
print('Actor loss:', behavior_metrics['actor_loss'])
print('Critic loss:', behavior_metrics['critic_loss'])
print('Imagined reward mean:', behavior_metrics['imagined_reward_mean'])
print('Imagined value mean:', behavior_metrics['imagined_value_mean'])
print('Actor parameters changed:', bool((actor_after - actor_before).abs().mean().item() > 0.0))
print('Critic parameters changed:', bool((critic_after - critic_before).abs().mean().item() > 0.0))
print('World-model parameters unchanged:', bool(torch.equal(world_before, world_after)))
print('All imagined returns finite:', bool(torch.isfinite(returns).all().item()))

Seed used: 7
Imagination horizon: 5
Imagined feature shape: (5, 4, 160)
Imagined action shape: (5, 4, 2)
Imagined reward shape: (5, 4, 1)
Imagined value shape: (5, 4, 1)
TD-lambda return shape: (5, 4, 1)
Actor loss: -0.4270327687263489
Critic loss: 1.4222424030303955
Imagined reward mean: 0.24776263535022736
Imagined value mean: -0.3628440201282501
Actor parameters changed: True
Critic parameters changed: True
World-model parameters unchanged: True
All imagined returns finite: True


## Alternating training pipeline smoke test

This cell runs one tiny collect → world-model-train → behavior-train → policy-collect cycle so the current M5 baseline is exercised end to end on a small replay buffer.

In [15]:
try:
    from tiny_dreamer_highway.models import TinyWorldModel, Actor, Critic
    from tiny_dreamer_highway.data.replay_buffer import ReplayBuffer
    from tiny_dreamer_highway.training import run_training_cycle
except ImportError:
    from tiny_dreamer_highway.models.world_model import TinyWorldModel
    from tiny_dreamer_highway.models.actor import Actor
    from tiny_dreamer_highway.models.critic import Critic
    from tiny_dreamer_highway.data.replay_buffer import ReplayBuffer
    from tiny_dreamer_highway.training.pipeline import run_training_cycle

set_global_seeds(config.seed, deterministic_torch=False)

latent_dim = config.model.deterministic_dim + config.model.stochastic_dim
pipeline_replay_buffer = ReplayBuffer(capacity=config.replay.capacity)
pipeline_world_model = TinyWorldModel(
    observation_shape=(observation_tensor.shape[1], observation_tensor.shape[2], observation_tensor.shape[3]),
    action_dim=action_tensor.shape[-1],
    embedding_dim=config.model.embedding_dim,
    deterministic_dim=config.model.deterministic_dim,
    stochastic_dim=config.model.stochastic_dim,
    hidden_dim=config.model.hidden_dim,
    rssm_num_layers=config.model.rssm_num_layers,
    reward_hidden_dim=config.model.reward_hidden_dim,
    reward_num_layers=config.model.reward_num_layers,
    )
pipeline_actor = Actor(
    latent_dim=latent_dim,
    action_dim=action_tensor.shape[-1],
    hidden_dim=config.model.actor_hidden_dim,
    num_layers=config.model.actor_num_layers,
)
pipeline_critic = Critic(
    latent_dim=latent_dim,
    hidden_dim=config.model.critic_hidden_dim,
    num_layers=config.model.critic_num_layers,
)
pipeline_world_optimizer = torch.optim.Adam(pipeline_world_model.parameters(), lr=config.training.world_model_lr)
pipeline_actor_optimizer = torch.optim.Adam(pipeline_actor.parameters(), lr=config.training.actor_lr)
pipeline_critic_optimizer = torch.optim.Adam(pipeline_critic.parameters(), lr=config.training.critic_lr)

pipeline_metrics = run_training_cycle(
    config,
    pipeline_replay_buffer,
    pipeline_world_model,
    pipeline_actor,
    pipeline_critic,
    pipeline_world_optimizer,
    pipeline_actor_optimizer,
    pipeline_critic_optimizer,
    warm_start_steps=16,
    policy_steps=4,
    seed=config.seed,
    )

print('Seed used:', config.seed)
print('Warm-start transitions added:', pipeline_metrics.warm_start_added)
print('Policy transitions added:', pipeline_metrics.policy_added)
print('Replay size after cycle:', pipeline_metrics.replay_size)
print('World-model metrics:', pipeline_metrics.world_model_metrics)
print('Behavior metrics:', pipeline_metrics.behavior_metrics)
print('Replay grew after policy collection:', bool(pipeline_metrics.replay_size > pipeline_metrics.warm_start_added))

Seed used: 7
Warm-start transitions added: 16
Policy transitions added: 4
Replay size after cycle: 20
World-model metrics: {'reconstruction_loss': 6202.5771484375, 'reconstruction_mse': 1.1907249689102173, 'observation_log_prob': -6202.5771484375, 'reward_loss': 5.721320152282715, 'continue_loss': 0.6493415832519531, 'kl_loss': 51.48883819580078, 'kl_loss_raw': 51.48883819580078, 'total_loss': 6260.4365234375}
Behavior metrics: {'actor_loss': -0.06494514644145966, 'critic_loss': 1.5369991064071655, 'imagined_reward_mean': 0.17331942915916443, 'imagined_value_mean': 0.31681564450263977}
Replay grew after policy collection: True


## Checkpoint save/load smoke test

This cell saves the current tiny pipeline state into the Drive-backed checkpoint folder, perturbs parameters in memory, reloads the checkpoint, and verifies that the saved weights and metadata are restored.

In [16]:
try:
    from tiny_dreamer_highway.training import save_checkpoint, load_checkpoint, find_latest_checkpoint
except ImportError:
    from tiny_dreamer_highway.training.checkpointing import (
        save_checkpoint,
        load_checkpoint,
        find_latest_checkpoint,
    )

checkpoint_dir = ARTIFACT_ROOT / 'checkpoints'
checkpoint_step = 1
checkpoint_metrics = {
    'world_total_loss': pipeline_metrics.world_model_metrics['total_loss'],
    'actor_loss': pipeline_metrics.behavior_metrics['actor_loss'],
    'critic_loss': pipeline_metrics.behavior_metrics['critic_loss'],
}

pipeline_world_before_save = next(pipeline_world_model.parameters()).detach().clone()
pipeline_actor_before_save = next(pipeline_actor.parameters()).detach().clone()
pipeline_critic_before_save = next(pipeline_critic.parameters()).detach().clone()

checkpoint_file = save_checkpoint(
    checkpoint_dir=checkpoint_dir,
    step=checkpoint_step,
    world_model=pipeline_world_model,
    actor=pipeline_actor,
    critic=pipeline_critic,
    world_model_optimizer=pipeline_world_optimizer,
    actor_optimizer=pipeline_actor_optimizer,
    critic_optimizer=pipeline_critic_optimizer,
    metrics=checkpoint_metrics,
    )

with torch.no_grad():
    next(pipeline_world_model.parameters()).add_(1.0)
    next(pipeline_actor.parameters()).add_(1.0)
    next(pipeline_critic.parameters()).add_(1.0)

checkpoint_metadata = load_checkpoint(
    checkpoint_file,
    world_model=pipeline_world_model,
    actor=pipeline_actor,
    critic=pipeline_critic,
    world_model_optimizer=pipeline_world_optimizer,
    actor_optimizer=pipeline_actor_optimizer,
    critic_optimizer=pipeline_critic_optimizer,
    map_location='cpu',
    )
latest_checkpoint = find_latest_checkpoint(checkpoint_dir)

pipeline_world_after_load = next(pipeline_world_model.parameters()).detach().clone()
pipeline_actor_after_load = next(pipeline_actor.parameters()).detach().clone()
pipeline_critic_after_load = next(pipeline_critic.parameters()).detach().clone()

print('Checkpoint file:', checkpoint_file)
print('Latest checkpoint:', latest_checkpoint)
print('Loaded step:', checkpoint_metadata['step'])
print('Loaded metrics:', checkpoint_metadata['metrics'])
print('World-model restored:', bool(torch.allclose(pipeline_world_before_save, pipeline_world_after_load)))
print('Actor restored:', bool(torch.allclose(pipeline_actor_before_save, pipeline_actor_after_load)))
print('Critic restored:', bool(torch.allclose(pipeline_critic_before_save, pipeline_critic_after_load)))

Checkpoint file: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/checkpoints/checkpoint_00001.pt
Latest checkpoint: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/checkpoints/checkpoint_00001.pt
Loaded step: 1
Loaded metrics: {'world_total_loss': 6260.4365234375, 'actor_loss': -0.06494514644145966, 'critic_loss': 1.5369991064071655}
World-model restored: True
Actor restored: True
Critic restored: True


## Metrics export smoke test

This cell exports the latest pipeline metrics into the Drive-backed logs folder as JSONL, CSV, and a latest-summary JSON file, then verifies that those artifact files exist.

In [17]:
try:
    from tiny_dreamer_highway.training import export_cycle_metrics
except ImportError:
    from tiny_dreamer_highway.training.metrics_logging import export_cycle_metrics

metrics_log_dir = ARTIFACT_ROOT / 'logs'
metrics_outputs = export_cycle_metrics(
    metrics_log_dir,
    step=checkpoint_metadata['step'],
    metrics=pipeline_metrics,
    checkpoint_file=checkpoint_file,
    )

print('JSONL metrics file:', metrics_outputs['jsonl'])
print('CSV metrics file:', metrics_outputs['csv'])
print('Summary metrics file:', metrics_outputs['summary'])
print('JSONL exists:', metrics_outputs['jsonl'].exists())
print('CSV exists:', metrics_outputs['csv'].exists())
print('Summary exists:', metrics_outputs['summary'].exists())

JSONL metrics file: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/logs/cycle_metrics.jsonl
CSV metrics file: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/logs/cycle_metrics.csv
Summary metrics file: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/logs/latest_summary.json
JSONL exists: True
CSV exists: True
Summary exists: True


## N-step prediction evaluation smoke test

This cell evaluates short imagined rollouts against actual replay sequence frames and reports MSE, PSNR, SSIM-style metrics, plus the Phase 3 observation negative log-likelihood from the probabilistic decoder.

In [18]:
try:
    from tiny_dreamer_highway.evaluation import evaluate_n_step_predictions
except ImportError:
    from tiny_dreamer_highway.evaluation.prediction_eval import evaluate_n_step_predictions

evaluation_horizon = min(3, sequence_observations.shape[1] - 1)
seed_observations = sequence_observations[:, 0]
future_actions = sequence_actions[:, :evaluation_horizon]
target_observations = sequence_observations[:, 1 : evaluation_horizon + 1]

evaluation_results = evaluate_n_step_predictions(
    pipeline_world_model,
    seed_observations,
    future_actions,
    target_observations,
    )

print('Evaluation horizon:', evaluation_horizon)
print('Prediction tensor shape:', tuple(evaluation_results['predictions'].shape))
print('Observation decoder std:', pipeline_world_model.decoder.distribution_std)
print('Summary metrics:', evaluation_results['summary'])
for step_metrics in evaluation_results['step_metrics']:
    line = (
        f"Step {int(step_metrics['step'])}: mse={step_metrics['mse']:.6f}, "
        f"psnr={step_metrics['psnr']:.3f}, ssim={step_metrics['ssim']:.4f}"
    )
    if 'nll' in step_metrics:
        line += f", nll={step_metrics['nll']:.3f}"
    print(line)

Evaluation horizon: 3
Prediction tensor shape: (4, 3, 1, 64, 64)
Observation decoder std: 1.0
Summary metrics: {'mse_mean': 0.130851740638415, 'psnr_mean': 8.834093902264703, 'ssim_mean': 0.008041519826898972, 'nll_mean': 4031.9563802083335}
Step 1: mse=0.125473, psnr=9.015, ssim=0.0080, nll=4020.941
Step 2: mse=0.132954, psnr=8.763, ssim=0.0075, nll=4036.262
Step 3: mse=0.134128, psnr=8.725, ssim=0.0086, nll=4038.667


## Plot artifact smoke test

This cell saves a metric curve plot and a predicted-versus-target comparison grid so the evaluation outputs become report-ready image artifacts.

In [19]:
try:
    from tiny_dreamer_highway.evaluation import export_prediction_artifacts
except ImportError:
    from tiny_dreamer_highway.evaluation.visualization import export_prediction_artifacts

media_dir = ARTIFACT_ROOT / 'media'
plot_outputs = export_prediction_artifacts(
    evaluation_results['step_metrics'],
    evaluation_results['predictions'],
    target_observations,
    media_dir,
    prefix='n_step_eval_smoke',
    max_steps=evaluation_horizon,
    )

print('Metrics plot:', plot_outputs['metrics_plot'])
print('Comparison grid:', plot_outputs['comparison_grid'])
print('Metrics plot exists:', plot_outputs['metrics_plot'].exists())
print('Comparison grid exists:', plot_outputs['comparison_grid'].exists())

Metrics plot: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/media/n_step_eval_smoke_metrics.png
Comparison grid: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/media/n_step_eval_smoke_comparison.png
Metrics plot exists: True
Comparison grid exists: True


## Video artifact smoke test

This cell exports a lightweight GIF that shows target frames, predicted frames, and absolute-error frames side by side so the evaluation results can be reviewed as a short rollout video.

In [20]:
try:
    from tiny_dreamer_highway.evaluation import export_prediction_video
except ImportError:
    from tiny_dreamer_highway.evaluation.visualization import export_prediction_video

video_path = export_prediction_video(
    evaluation_results['predictions'],
    target_observations,
    media_dir / 'n_step_eval_smoke_comparison.gif',
    max_steps=evaluation_horizon,
    fps=2,
    )

print('Comparison video:', video_path)
print('Video exists:', video_path.exists())
print('Video size (bytes):', video_path.stat().st_size)

Comparison video: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/media/n_step_eval_smoke_comparison.gif
Video exists: True
Video size (bytes): 23707


## Submission bundle smoke test

This cell packages the generated evaluation artifacts into a manifest-backed folder and a zip archive so the latest outputs are ready for final submission handoff.

In [21]:
try:
    from tiny_dreamer_highway.evaluation import export_submission_bundle
except ImportError:
    from tiny_dreamer_highway.evaluation.artifact_bundle import export_submission_bundle

bundle_outputs = export_submission_bundle(
    {
        'metrics_plot': plot_outputs['metrics_plot'],
        'comparison_grid': plot_outputs['comparison_grid'],
        'comparison_video': video_path,
        'metrics_summary': metrics_outputs['summary'],
        'checkpoint': checkpoint_file,
    },
    ARTIFACT_ROOT / 'bundles',
    bundle_name='tiny_dreamer_submission_bundle',
    metadata={
        'evaluation_horizon': int(evaluation_horizon),
        'checkpoint_step': int(checkpoint_metadata['step']),
        'project': 'CSC_580_Final_Project',
    },
    )

print('Bundle directory:', bundle_outputs['bundle_dir'])
print('Bundle manifest:', bundle_outputs['manifest'])
print('Bundle archive:', bundle_outputs['archive'])
print('Bundle archive exists:', bundle_outputs['archive'].exists())

Bundle directory: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/bundles/tiny_dreamer_submission_bundle
Bundle manifest: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/bundles/tiny_dreamer_submission_bundle/manifest.json
Bundle archive: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/bundles/tiny_dreamer_submission_bundle.zip
Bundle archive exists: True
